In [0]:
%sql
-- Gold: company news at article-level granularity, with story count and days-covered
-- added as summary columns per symbol (not collapsed/aggregated).
 
-- Step 1: Define the table schema.
CREATE OR REPLACE TABLE gold_finnhub_news_summary
(
  symbol STRING COMMENT "Stock ticker symbol the news relates to.",
  news_id BIGINT COMMENT "Unique identifier for the news article.",
  headline STRING COMMENT "Headline of the news article.",
  source STRING COMMENT "Publisher or news source.",
  published_at TIMESTAMP COMMENT "Timestamp when the article was published.",
  category STRING COMMENT "Category assigned to the article.",
  summary STRING COMMENT "Short summary of the article.",
  url STRING COMMENT "Link to the full article.",
  total_stories BIGINT COMMENT "Total number of distinct news articles for this symbol.",
  days_covered INT COMMENT "Number of days between the earliest and latest article for this symbol."
)
COMMENT "Article-level company news with symbol-level coverage stats attached, for the finnhub medallion pipeline.";
 
-- Step 2: Populate it.
INSERT OVERWRITE TABLE gold_finnhub_news_summary
SELECT
  symbol,
  news_id,
  headline,
  source,
  published_at,
  category,
  summary,
  url,
  COUNT(news_id) OVER (PARTITION BY symbol) AS total_stories,
  DATEDIFF(
    MAX(published_at) OVER (PARTITION BY symbol),
    MIN(published_at) OVER (PARTITION BY symbol)
  ) AS days_covered
FROM silver_finnhub_company_news;
 

In [0]:
%sql
SELECT SYMBOL, COUNT(*) FROM gold_finnhub_news_summary GROUP BY 1